In [1]:
import os
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

c:\Users\tarus\anaconda3\envs\gpu_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

KB_PATH = r"C:\Users\tarus\minor_project\Crop_Disease_Prediction\KB"

documents = []
metadata = []

# load all JSON files
for file in os.listdir(KB_PATH):
    if file.endswith(".json"):
        with open(os.path.join(KB_PATH, file), "r", encoding="utf-8") as f:
            data = json.load(f)

            # each crop JSON contains {"crop": name, "diseases": [...]}
            for crop in data:
                crop_name = crop["crop"]
                for disease in crop["diseases"]:
                    text = f"Crop: {crop_name}\nDisease: {disease['name']}\nSymptoms: {disease.get('symptoms', [])}\nManagement: {disease.get('management', {})}"

                    documents.append(text)
                    metadata.append({
                        "crop": crop_name,
                        "disease": disease["name"],
                        "url": disease["url"]
                    })

In [3]:
len(documents)

131

In [4]:
# convert to embeddings
embeddings = embedder.encode(documents, convert_to_numpy=True)

# create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

In [5]:
def query_disease(user_query, k=3):
    # encode query
    q_emb = embedder.encode([user_query], convert_to_numpy=True)

    # search top-k
    dist, indices = index.search(q_emb, k)
    retrieved_docs = [documents[i] for i in indices[0]]

    return retrieved_docs, dist, indices

In [6]:
# example query
retrieved_docs, dist, indices = query_disease("what are the symptoms and management practices of Wheat Stem Fly?")

In [11]:
print(f"Remedies: {retrieved_docs}")
print(f"Nearest neighbour: {dist}")

Remedies: ["Crop: Wheat\nDisease: Loose Smut:\nSymptoms: ['The entire inflorescence, except the rachis, is replaced by masses of smut spores.', 'These black teliospores often are blown away by the wind, leaving only the bare rachis and remnants of other floral structures.']\nManagement: {'management_practices': ['Treat the seed with Vitavax @ 2g/kg seed before sowing.', 'Burry the infected ear heads in the soil, so that secondary spread is avoided.']}", "Crop: Maize\nDisease: Pink stem borer: Sesamia inferens\nSymptoms: ['Pink larva damage stem leading to dead heart', 'Bead like eggs laid in rows within leaf sheath', 'Pinkish brown larvae with dark head', 'Straw coloured moths with white wings']\nManagement: {'management_practices': ['When infestation crosses 10%, spray phosalone 35EC 1150 ml/ha or chlorantraniliprole 18.5 SC @150ml/ha twice at 20 days interval']}", "Crop: Maize\nDisease: Stem borer: Chilo partellus\nSymptoms: ['Central shoot dries leading to dead heart', 'Young larva 

In [12]:
import ollama

In [54]:
def generate_hindi_answer_ollama(disease_or_pest_name, retrieved_docs, model_name="llama3"):
    """
    Generate a farmer-friendly answer in pure Hindi using Ollama (LLaMA 3),
    for a pest or disease.
    """
    context = "\n\n---\n\n".join(retrieved_docs)

    prompt = f"""
    You are an agricultural advisor. A farmer asked about: "{disease_or_pest_name}" (pest or disease).  
    Use this context if relevant: {context}.  

    Give the reply in two versions- one pure Hindi and another one pure English, but keep chemical names, tools, and numbers in English for Hindi version. Also, the English version should be simpler to understand by farmers  
    Structure answer in two parts:  
    1. Symptoms  
    2. Management  

    Provide the answer in farmer-friendly and easy to understand language.
    If exact info not found, generate best possible symptoms & management in the same format as mentioned above.
    """
    response = ollama.chat(model=model_name, messages=[{"role": "user", "content": prompt}])

    hindi_answer = response['message']['content']
    return hindi_answer

In [65]:
query = "Coffee Rust"

retrieved_docs, dist, indices = query_disease(query)

In [66]:
retrieved_docs

["Crop: Coffee\nDisease: Die back or Anthranos: Collectorichum coffeasum\nSymptoms: ['On leaves circular to grayish spots of 2-3 m in dia.', 'On berries small dark coloured sunken spots are farmed. Beans become brown. Die back also occurs.']\nManagement: {'management_practices': ['Spraying Mancozeb 0.25%']}",
 "Crop: Coffee\nDisease: Rust: Hemileia vastatrix\nSymptoms: ['It mostly affects  leaves or some time leaves and tender shoots. In earlier stage, 2- 3m.m size yellow spots develop on the lower surface of leaves.', 'Rust lesions develop as orange yellow spots on lower surface of leaves. Thousasnds of small rust appear on the leaf. The upper surface of the leaves become yellow to brown.', 'At later stages, yellow rusty appearances are found on the entire undersurface of the leaf. The orange yellow spots are followed by black spots followed by necrosis.', 'Infected leaves defoliate and branch remains without any leaves. The plants remain stunted with reduced yield. Reduced yileds and

In [67]:
dist

array([[0.9483911, 0.9727299, 1.0181723]], dtype=float32)

In [68]:
threshold = 1.0  # tune based on embedding similarity

if dist[0][0] < threshold:
    # Relevant documents found → use them as context
    hindi_answer = generate_hindi_answer_ollama(query, retrieved_docs)
else:
    # No reliable match → ask LLM to generate own response
    hindi_answer = generate_hindi_answer_ollama(query, retrieved_docs=[])


In [64]:
print(hindi_answer)

Here are the answers in both Hindi and English:

**Sugarcane Red Rot**

**Hindi:**
1. लक्षण:
सग्ग्रे सागान की पत्तियां हरे रंग से橙 और फिर पीले रंग में बदल जाती हैं,third or fourth leaf में। इसके बाद पत्तियों का निचला हिस्सा शुरू से ऊपर की ओर सूखने लगता है।
पतझड़ के पीछे की तरफ लाल रंग के दाग देखे जाते हैं, यदि फंगल स्पोर पतझड़ के मध्यरिब से पतझड़ के पिछले हिस्से में प्रवेश करता है।
पूरे सागान को 10 दिनों बाद सूखने लगता है।
जब प्रभावित सागान को खोला जाता है, तो उसके अंदर की parte reddish in color होती है और intermittently white tinges होती हैं।
कभी-कभी सागान के पिथ में काले ब्राउन तरल और अल्कोहल से संबंधित गंध देखी जाती है।
पतझड़ के मध्यरिब पर लाल रंग का दाग देखा जाता है।

2. प्रबंधन:
पथोगेन: सग्ग्रे सागान की लाल रोट बीमारी के कारण fungus Glomerella tucumanensis है। कुछ पाथोलॉजिस्त्स उसके एक older name, Colletotrichum falcatum, स्टिल पREFER किया करते हैं।

प्रबंधन पрак्रियाएं:
स्वस्थ plant से लिए गए setts को सागान की बीमारी से मुक्त área में लगाया जाता है।
प्रभावित खेत को rice से एक सीज

**Translate into Hindi first and then store in FAISS DB**

In [7]:
from transformers import MarianMTModel, MarianTokenizer

In [8]:
model_name = "Helsinki-NLP/opus-mt-en-hi"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

c:\Users\tarus\anaconda3\envs\gpu_env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tarus\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-en-hi. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\tarus\anaconda3\envs\gpu_env\lib\site-packages\transformers\models\marian\tokenization_marian.p

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [9]:
def translate_to_hindi(text):
    batch = tokenizer([text], return_tensors="pt", padding=True)
    translated = model.generate(**batch)
    hindi_text = tokenizer.decode(translated[0], skip_special_tokens=True)
    return hindi_text

In [10]:
# Example
english_text = "Wheat aphids cause yellowing of leaves."
print(translate_to_hindi(english_text))

समुद्री डाकू पत्ते के पीले रंग का कारण होते हैं.
